# DeepSeek-OCR Q8 Test (Pure Grounding Mode)

This version removes the system prompt to allow the `<|grounding|>` trigger to work without interference.


In [2]:
import base64
import time
import requests
from openai import OpenAI

# 1. Setup Client
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# 2. Detect Model ID
try:
    model_list = requests.get("http://localhost:1234/v1/models").json()
    active_model = model_list["data"][0]["id"]
    print(f"✅ Detected: {active_model}")
except:
    active_model = "deepseek-ocr-q8"


def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


img_path = "ollama_test_table.png"
base64_image = encode_image(img_path)
print("Setup complete.")

✅ Detected: deepseek-ocr@bf16
Setup complete.


In [3]:
print(f"Requesting inference...\n")
start_time = time.time()

response = client.chat.completions.create(
    model=active_model,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "<|grounding|>Convert the document to markdown.",
                },
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{base64_image}"},
                },
            ],
        }
    ],
    max_tokens=4096,
    temperature=0,
    stream=True,
)

full_response = ""
for chunk in response:
    if chunk.choices[0].delta.content:
        content = chunk.choices[0].delta.content
        print(content, end="", flush=True)
        full_response += content

print(f"\n\n--- Finished in {time.time() - start_time:.2f} seconds ---")

Requesting inference...

<table><tr><td></td><td>النقش الواجب خصمها</td><td>اللجنة</td><td>الرقم التربني</td></tr><tr><td>6</td><td>...</td><td>...</td><td>01</td></tr><tr><td>2</td><td>...</td><td>...</td><td>07</td></tr><tr><td>18</td><td>...</td><td>...</td><td>13</td></tr></table>

--- Finished in 12.85 seconds ---
